# 10: Classifier Evaluation

**Author:** Dr Rob Lyon 

**Version:** 1.0

## Code & License
Copyright © 2026 Robert Lyon. All Rights Reserved.

This notebook and all associated materials are the intellectual property of the author.

Permission is granted solely to read, study, and analyse this material for personal educational purposes. No other rights are granted.

Without the prior written consent of the author, you may not:

* Copy, reproduce, redistribute, publish, transmit, or display this work in whole or in part.
* Modify, adapt, transform, translate, or create derivative works based on this material.
* Incorporate any portion of this work into another project, publication, product, model, dataset, or codebase.
* Use this material for commercial purposes.
* Remove or alter this copyright notice.

All intellectual property rights, including copyright and any derivative rights, remain exclusively vested in the author.

Access to this material does not constitute a transfer of ownership, license, or any other intellectual property rights except as expressly stated above.

---

**Note:**

This notebook is designed to be viewed from within JupyterLab. The Table of Contents and "Back to Table of Contents" links rely on JupyterLab's anchor handling to jump between sections.

If you open this notebook in another tool, such as PyCharm or Visual Studio Code, these anchor links may not work as expected. You can still navigate the notebook in those tools by using the headings directly, most editors provide an outline or contents panel that lists them for you.


---
## Table of Contents
1. [Introduction](#1.-Introduction)
2. [Useful Links](#2.-Useful-Links)
3. [Libraries and Environment Setup](#3.-Libraries-and-Environment-Setup)
4. [Why Evaluation Is Hard](#4.-Why-Evaluation-Is-Hard)
    - [4.1 The Memorisation Problem](#4.1-The-Memorisation-Problem)
    - [4.2 The i.i.d. Assumption](#4.2-The-i.i.d.-Assumption)
    - [4.3 The No Free Lunch Theorem](#4.3-The-No-Free-Lunch-Theorem)
5. [The Experimental Pipeline: Keep Test Conditions Identical](#5.-The-Experimental-Pipeline:-Keep-Test-Conditions-Identical)
6. [How Much Data for Training vs Testing?](#6.-How-Much-Data-for-Training-vs-Testing?)
7. [Class Distributions and Sampling](#7.-Class-Distributions-and-Sampling)
    - [7.1 The Class Imbalance Problem](#7.1-The-Class-Imbalance-Problem)
    - [7.2 Sampling Methods](#7.2-Sampling-Methods)
    - [7.3 Weighted Sampling: Deliberately Rebalancing](#7.3-Weighted-Sampling:-Deliberately-Rebalancing)
8. [K-Fold Cross Validation](#8.-K-Fold-Cross-Validation)
    - [8.1 K-Fold vs Single Split: Stability](#8.1-K-Fold-vs-Single-Split:-Stability)
    - [8.2 Choosing k: the Bias-Variance Trade-Off](#8.2-Choosing-k:-the-Bias-Variance-Trade-Off)
9. [The Confusion Matrix](#9.-The-Confusion-Matrix)
    - [9.1 Binary Confusion Matrix](#9.1-Binary-Confusion-Matrix)
    - [9.2 Binary Confusion Matrix with Seaborn](#9.2-Binary-Confusion-Matrix-with-Seaborn)
    - [9.3 Multi-Class Confusion Matrix](#9.3-Multi-Class-Confusion-Matrix)
    - [9.4 Why Accuracy Lies on Imbalanced Data](#9.4-Why-Accuracy-Lies-on-Imbalanced-Data)
10. [Summary](#10.-Summary)
11. [References](#11.-References)


---

## 1. Introduction

This notebook covers **classifier evaluation**. Building a model is straightforward; knowing whether it actually works, and whether it is better than alternatives, requires careful experimental discipline that is easy to get wrong.

We begin with why evaluation is hard and what can go wrong: the memorisation problem, the i.i.d. assumption, and the No Free Lunch theorem. We then build up the correct experimental pipeline step by step: how to create a good test set, why class distributions matter, how different sampling strategies affect reliability, and why k-fold cross-validation gives more trustworthy estimates than a single random split. We close by introducing the **confusion matrix**, the foundation of all classification metrics, for both binary and multi-class problems, and show concretely how accuracy alone can be deeply misleading on imbalanced data.

### Learning Objectives

By the end of this notebook you should be able to:

- Explain why evaluating a model on training data is always wrong, and define the generalisation gap.
- State the i.i.d. assumption and describe practical scenarios in which it is violated.
- Explain the No Free Lunch theorem and its implication for model comparison.
- Construct a correct experimental pipeline: one fixed split, one fitted scaler, one test set for all models.
- Discuss the trade-off between training set size and test set reliability, and justify a choice of split ratio.
- Distinguish random, stratified, and weighted sampling and explain when each is appropriate.
- Implement k-fold stratified cross-validation and interpret the mean and standard deviation of CV scores.
- Explain the bias-variance trade-off in choosing $k$, and justify the use of $k = 5$ or $k = 10$.
- Construct and interpret binary and multi-class confusion matrices.
- Calculate accuracy from a confusion matrix and explain why accuracy alone is insufficient on imbalanced data.

---


## 2. Useful Links

| Link | Description |
|------|-------------|
| [scikit-learn: cross-validation](https://scikit-learn.org/stable/modules/cross_validation.html) | Comprehensive guide to cross-validation strategies in scikit-learn, including stratified k-fold, leave-one-out, and group-based splits. Essential reading alongside this notebook. |
| [scikit-learn: model evaluation](https://scikit-learn.org/stable/modules/model_evaluation.html) | Overview of all scoring metrics available in scikit-learn. Covers accuracy, precision, recall, F1, ROC-AUC and more; useful preparation for Notebooks 11 and 12. |
| [scikit-learn: confusion_matrix](https://scikit-learn.org/stable/modules/generated/sklearn.metrics.confusion_matrix.html) | API reference for `confusion_matrix`, including how to use `normalize` to produce proportions rather than counts. |
| [scikit-learn: train_test_split](https://scikit-learn.org/stable/modules/generated/sklearn.model_selection.train_test_split.html) | API reference for `train_test_split`. Pay particular attention to the `stratify` parameter introduced in Section 7. |
| [No Free Lunch Theorems (Wolpert & Macready, 1997)](https://ieeexplore.ieee.org/document/585893) | The original paper establishing the No Free Lunch theorems for optimisation and supervised learning. Technically demanding but important background reading. |


---

## 3. Libraries and Environment Setup

### 3.1 Working Environment

To run the code in this notebook, you will need a Python environment with the required libraries listed below installed.

The easiest way to get started is to use the **Project IO** Docker container, which provides a fully configured and tested environment containing all required Python packages and supporting dependencies. Instructions for downloading and running the container can be found in the **project-io** GitHub repository:

https://github.com/scienceguyrob/project-io

Using the Docker container ensures that the notebooks run in exactly the same environment in which they were developed and tested.

If you prefer not to use Docker, a good alternative is to install the [Anaconda Distribution](https://www.anaconda.com/products/distribution). You can then create a new Python environment and install the required libraries using either `conda install` or `pip install`.

### 3.2 Libraries Used in This Notebook

All sections of this notebook require the external libraries listed below. No built-in-only sections exist in this notebook.

| Library | Purpose | Documentation |
|---------|---------|---------------|
| **NumPy** (`numpy`) | Numerical arrays, random number generation, and mathematical operations. | [numpy.org](https://numpy.org/doc/stable/) |
| **pandas** (`pandas`) | Tabular data manipulation. Used to build result summary tables. | [pandas.pydata.org](https://pandas.pydata.org/docs/) |
| **Matplotlib** (`matplotlib.pyplot`) | Standard Python plotting library. Used throughout for line plots, bar charts, and visual diagnostics. | [matplotlib.org](https://matplotlib.org/stable/) |
| **Matplotlib patches** (`matplotlib.patches`) | Used to draw coloured rectangles in the k-fold visualisation. | [matplotlib.org/patches](https://matplotlib.org/stable/api/patches_api.html) |
| **Seaborn** (`seaborn`) | Statistical visualisation built on Matplotlib. Used for annotated confusion matrix heatmaps. | [seaborn.pydata.org](https://seaborn.pydata.org/) |
| **scikit-learn** (`sklearn`) | The primary machine learning library. Provides datasets, classifiers, evaluation tools, and preprocessing utilities used throughout this notebook. | [scikit-learn.org](https://scikit-learn.org/stable/) |
| **ipywidgets** (`ipywidgets`) | Adds interactive UI controls (sliders, dropdowns, etc.) to Jupyter notebooks. We use it to create sliders for adjusting plot parameters in real time. | [ipywidgets.readthedocs.io](https://ipywidgets.readthedocs.io/en/stable/) |
| **ipympl** (`matplotlib widget` backend) | Enables interactive, live-updating Matplotlib figures inside Jupyter. Required for smooth slider updates without redrawing the entire plot. | [matplotlib.org/ipympl](https://matplotlib.org/ipympl/) |

### 3.3 Imports

All library imports for this notebook are placed in the cell below. This is a deliberate best practice: keeping all imports in one place makes it easy to see at a glance what is required and avoids confusing errors caused by skipped cells.

> ⚠️ **You must run the cell below before executing any other code in this notebook.** If you skip this cell, all subsequent code cells will raise a `NameError`.

In [ ]:
# Listing 1.
# ─────────────────────────────────────────────────────────────────────────────
# All library imports for this notebook are placed here for clarity.
# Run this cell first before executing any code.
# ─────────────────────────────────────────────────────────────────────────────
import numpy as np                                        # Numerical arrays and mathematical operations
import pandas as pd                                       # Tabular data manipulation
import matplotlib.pyplot as plt                           # Plotting and visualisation
import matplotlib.patches as mpatches                     # Rectangle patches for custom diagrams
import seaborn as sns                                     # Statistical visualisation (heatmaps, etc.)
from collections import Counter                           # Counting occurrences of values

import matplotlib
matplotlib.rcParams['figure.max_open_warning'] = 50

# scikit-learn datasets
from sklearn.datasets import make_classification, load_breast_cancer, load_iris, make_moons

# scikit-learn classifiers
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier

# scikit-learn model selection
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score

# scikit-learn evaluation and preprocessing
from sklearn.metrics import accuracy_score, confusion_matrix
from sklearn.preprocessing import StandardScaler

# Reproducible random number generation
rng = np.random.default_rng(42)

# Confirm successful import
print('Libraries loaded successfully.')


---

## 4. Why Evaluation Is Hard

🔙 [Back to Table of Contents](#table-of-contents)

It is tempting to think that once a model is trained, the hard part is over and all that remains is to check how well it performs. In reality, the opposite is usually true. Training a model is often the easy part; *evaluating* it correctly is where most mistakes happen. A model that looks excellent on paper can be worthless in the real world if it was evaluated incorrectly, while a model that looks mediocre might actually be the better choice once it is tested fairly. Every evaluation mistake boils down to the same underlying error: accidentally answering the question "how well did the model memorise the data it was trained on?" when the question that actually matters is "how will this model behave on data it has never encountered?".

Before introducing any evaluation technique, it is worth pausing to ask: what could go wrong? Three foundational issues make evaluation harder than it first appears, and understanding them now will motivate every design decision in the experimental pipeline that follows in the rest of this notebook. 

The first is the **memorisation problem**, the risk that a model appears to perform well simply because it has seen the answers before. The second is the **i.i.d. assumption**, a quiet but essential assumption about how data is generated, which, when violated, can make even a carefully evaluated model fail in practice. The third is the **No Free Lunch theorem**, a result that explains why there is no such thing as a universally "best" model, and why fair comparison between models matters so much.

### 4.1 The Memorisation Problem

The very first instinct when evaluating a model is to measure its accuracy only on the data it was trained on. This is always a mistake, for reasons that become clearer once we think about what a model is actually capable of doing.

A sufficiently complex model can **memorise** training data with ease, effectively storing every example together with its label without ever learning a pattern that would generalise to new, unseen examples. This phenomenon is called **overfitting**. A model that has overfit will often achieve near-perfect accuracy on the data it was trained on, while performing dramatically worse, sometimes barely better than random guessing, on data it has never seen before. Evaluating a model on its training data is a bit like giving someone the exact exam questions and the answers in advance, letting them memorise the answer sheet, and then judging how well they understand the subject based on how they perform on that *same* exam. A high score tells you they can remember the sheet well, not that they understood the material or could handle a different exam on the same topic.

This gap between "performance on data the model has seen" and "performance on data the model has not seen" is so important that it has its own name: the **generalisation gap**, defined as the difference between training accuracy and test accuracy. A small gap means the model has learned patterns that hold up beyond the specific examples it was trained on; it has *generalised*. A large gap means the model has, to some degree, memorised the training data rather than learning from it, and its training accuracy is therefore an unreliable, usually wildly optimistic, estimate of how it will perform in practice.

The code below makes this concrete using a Decision Tree.

> 📓 **Notebook 6 recap:** Decision trees were introduced in Notebook 6 as a model that splits the feature space into regions using a sequence of threshold rules. One detail is especially relevant here: a decision tree's `max_depth` parameter controls how many of these splits it is allowed to make. A shallow tree (small `max_depth`) can only draw a few, coarse boundaries, while a deep tree (large `max_depth`) can keep splitting until, eventually, every individual training point sits in its own tiny region. That extreme case is memorisation in its purest form: a rule so specific that it fits the training data perfectly but captures noise rather than signal.

In the code (in the file `visualisations/Notebook_10/Figure_1.py`) we train Decision Trees at every depth from 1 to 20 and, at each depth, measure two separate accuracies: 

1. one on the training data (the data the tree learned from).
2. one on a held-out test set (data the tree has never seen).

For clarity, the code we use to build the trees looks like this. We will walk through it one piece at a time.

---

**1. Dataset Creation**

```python
# ── Dataset ──────────────────────────────────────────────────────────────────
# A synthetic classification dataset is used so that the underlying signal is
# known and controllable. 10 features are generated in total: 5 of these
# carry genuine information about the class label ("informative"), and 2 are
# redundant — linear combinations of the informative features, included to
# make the problem more realistic (real datasets rarely have purely
# independent features). The remaining 3 features are pure noise.
X_base, y_base = make_classification(
    n_samples=500,       # total number of samples
    n_features=10,       # total number of features
    n_informative=5,     # features that actually carry signal
    n_redundant=2,       # features that are linear combinations of informative ones
    random_state=0,      # fixed seed so the figure is reproducible across runs
)
```

This first block builds the dataset. `make_classification` is a convenience function provided by scikit-learn, that generates a synthetic dataset with a known, controllable structure, rather than relying on a real-world dataset where the "true" relationship between features and labels is unknown. Here, 10 features are generated in total, but only 5 of them (`n_informative=5`) actually carry information that determines the class label; the rest are either redundant combinations of those informative features (`n_redundant=2`) or pure noise. This matters because it means we know, by construction, that a model which performs well is picking up on genuine signal rather than getting lucky, and it gives us a dataset realistic enough to demonstrate overfitting clearly without needing thousands of samples. The `random_state=0` simply fixes the random number generator so that the same dataset is produced every time this code is run, which keeps the figure reproducible.

**2. Training and Test Data Splitting**

```python
# A single 75/25 train/test split is used throughout. This is deliberate:
# the point of this figure is to isolate the effect of model COMPLEXITY
# (controlled by max_depth) on the generalisation gap, not the effect of
# the split itself — that is the subject of Task 1 later in this notebook.
X_tr, X_te, y_tr, y_te = train_test_split(
    X_base, y_base, test_size=0.25, random_state=0   # 75/25 split
)
```

The dataset is then split once into a training set (`X_tr`, `y_tr`) and a test set (`X_te`, `y_te`) using a 75/25 ratio, and this same split is reused for every tree depth in the experiment. This is a deliberate design choice: if the split changed at every depth, any differences we observed between depths could be partly due to the data changing rather than the model changing, which would muddy the comparison. By fixing the split, the only thing that varies as the experiment progresses is `max_depth`, so any change in the curves can be attributed entirely to model complexity, which is exactly the variable this figure is designed to explore.

**3. Experiment Setup**

```python
# ── Train a Decision Tree at every depth from 1 to 20 ───────────────────
# max_depth controls how many sequential splits the tree is permitted to
# make. A shallow tree (small max_depth) can only draw a few, coarse
# decision boundaries. A deep tree (large max_depth) can keep splitting
# until, in the extreme, every individual training point sits in its own
# tiny region — memorisation in its purest form.
depths = list(range(1, 21))
train_accs = []
test_accs = []
```

This sets up the experiment. `depths` is simply the list of tree depths we are going to try, from 1 up to and including 20. The two empty lists, `train_accs` and `test_accs`, will be filled in as the loop below runs, one training accuracy and one test accuracy for each depth in `depths`.

**4. Execute Experiment**

```python
for d in depths:
    dt = DecisionTreeClassifier(max_depth=d, random_state=0)
    dt.fit(X_tr, y_tr)
    # Accuracy on the data the tree learned from — this is the figure
    # the "exam questions in advance" instinct relies on, and it is
    # always optimistic.
    train_accs.append(accuracy_score(y_tr, dt.predict(X_tr)))
    # Accuracy on data the tree has never seen — this is the figure
    # that actually tells us how the tree will perform in the real world.
    test_accs.append(accuracy_score(y_te, dt.predict(X_te)))
```

This is the heart of the experiment. For each value of `d` in turn, a brand new Decision Tree is created with `max_depth=d` and trained from scratch on the training data, so by the time the loop finishes, 20 completely separate trees have been trained, one per depth. For each of these trees, two accuracy scores are calculated using `accuracy_score`: one comparing the tree's predictions on `X_tr` against the true training labels `y_tr` (training accuracy), and one comparing its predictions on `X_te` against the true test labels `y_te` (test accuracy). Each pair of scores is appended to `train_accs` and `test_accs` respectively, so that after the loop completes, both lists contain 20 values, one for each depth.

**5. Find Best Depth**

```python
# The depth that gives the best TEST accuracy is the depth you would
# actually want to deploy, even though it is not the depth with the
# highest training accuracy.
best_test_depth = depths[int(np.argmax(test_accs))]
```

Once the loop has finished, `np.argmax(test_accs)` finds the *position* in the `test_accs` list where the highest test accuracy occurs, and `depths[...]` converts that position back into the actual depth value it corresponds to. This single number, `best_test_depth`, represents the most reliable choice of model from this experiment: it is the depth that performed best on data the tree had never seen, which is the only kind of performance that matters once the model is deployed.

---

Watch what happens to the two curves as depth increases: training accuracy climbs steadily towards 100%, while test accuracy rises initially, as the tree learns genuinely useful splits, but then peaks and begins to fall as the tree starts carving out rules that fit noise specific to the training set rather than patterns that generalise. The widening gap between the two curves *is* the generalisation gap, and it is the visual signature of overfitting. It can be tempting to pick whichever model configuration achieves the highest training accuracy, but the figure below shows exactly why this is dangerous: training accuracy is monotonically increasing with model complexity here, and would lead you to pick the most complex, and most overfit, tree available, which is the worst possible choice for real-world performance.

In the figure produced by the code below, look for three things: the steelblue training accuracy curve climbing towards 100% as depth increases, the red test accuracy curve rising initially, reaching a peak, and then declining, and the shaded region between the two curves growing wider as depth increases. This shaded area is the generalisation gap made visible. The dotted vertical line marks the depth at which test accuracy is highest, which is the depth you would actually want to deploy, not the deepest tree, even though the deepest tree has the highest training accuracy.

In [ ]:
# Listing 2.
%matplotlib widget
from visualisations.Figure_1 import show
show()

---

What does the output show? At the maximum depth of 20, the tree has become complex enough to classify every single training example correctly, giving a training accuracy of 100%. If you judged this model only on this number, you would conclude it is a perfect classifier. However, the test accuracy tells a very different story: on data the tree has never seen, it gets only 80% of predictions correct. 

The 20 percentage point difference between these two figures is the **generalisation gap**, and it is the clearest possible illustration of memorisation. The tree at depth 20 has not learned a general rule for separating the two classes; it has learned the training set itself, including whatever noise and quirks happen to be present in those particular 375 examples. When presented with new examples that contain different noise, that memorised detail is no longer useful, and accuracy drops accordingly.

The wider lesson extends well beyond this one figure. Training accuracy can always be pushed towards 100% simply by making a model more complex, regardless of whether that complexity reflects anything real about the relationship between features and labels. For this reason, training accuracy on its own should never be treated as evidence that a model is good. Test accuracy, measured on data the model had no part in learning from, is the only number that tells you how the model is likely to behave once it is deployed and starts making predictions on genuinely new data.

---

### 4.2 The i.i.d. Assumption

The previous section established that test accuracy, not training accuracy, is what tells us how a model will perform in the real world. But this only holds if the test set is actually representative of the real world the model will eventually be used in. If the test set differs from the conditions the model will face once deployed, a high test accuracy can be just as misleading as a high training accuracy, for a different reason.

For test accuracy to be a valid estimate of real-world performance, the data must satisfy a property known as **i.i.d.**, short for *independently and identically distributed*. This is a statistical assumption that sits quietly underneath almost every evaluation technique in machine learning. It is worth understanding precisely what each half of the term means, because the two halves can fail in different ways and for different reasons.

* **Identically distributed** means that the training data and the test data are drawn from the same underlying population, under the same conditions. In practice, this means the same source, the same measurement process, and the same time period. If a model is trained on patient data from one hospital and tested on patient data from a different hospital, with a different demographic mix, different equipment, or different clinical practices, the two datasets come from different distributions. A high test accuracy in that situation tells you something about how the model performs on that second hospital's data, but it does not tell you how the model will perform on your actual target population, because the test set was never representative of it in the first place.

* **Independent** means that each individual data point is sampled without depending on, or sharing information with, any other data point. This is easy to violate without realising it. Taking ten repeated measurements from the same person, for example, produces ten rows in a dataset, but those ten rows are not ten independent pieces of evidence about people in general. They all share whatever is specific to that one person, so the effective amount of new information is closer to one data point than to ten. If some of those rows end up in the training set and others in the test set, the model is not really being tested on someone new; it has effectively already seen that person during training.

A useful way to think about the i.i.d. assumption is to ask, "if I collected a brand new batch of real-world data tomorrow, would it look statistically like my test set?" If the answer is no, because the population has changed, the conditions have changed, or the new batch would contain people or situations the model has already encountered, then the i.i.d. assumption is violated, and the test accuracy is measuring something other than real-world performance.

> ⚠️ **Warning:** Violations of the i.i.d. assumption are easy to introduce accidentally and easy to miss, because the resulting test accuracy still looks like a normal number. There is no warning sign in the metric itself. A few common ways this happens in practice are worth keeping in mind. Training on data from one time period and testing on data from a later period can silently fail if the underlying conditions have changed between the two periods, a phenomenon often called distribution shift. Splitting data at the row level rather than at the individual or group level can allow the same person, sensor, or session to appear in both the training and test sets, so the model is partially tested on data it has effectively already seen. Finally, if the data was collected in a particular order, for example chronologically or grouped by source, splitting it without shuffling first can produce a training set and test set that look systematically different from each other, even though they come from the same overall population.

---

The code below (see `visualisations/Notebook_10/Figure_2.py`) shows what happens when the i.i.d. assumption is violated by training a model on data drawn from one distribution and testing it on data drawn from a different one. We will walk through how this experiment is constructed before looking at the results.

**1. Dataset Creation**

```python
rng_iid = np.random.default_rng(7)
# Population A: centred at [2, 2] with positive correlation (training data)
X_popA = rng_iid.multivariate_normal([2, 2], [[1, 0.5], [0.5, 1]], 300)
y_popA = (X_popA[:, 0] + X_popA[:, 1] > 4).astype(int)

# Population B: centred at [5, 5] with negative correlation (test data)
X_popB = rng_iid.multivariate_normal([5, 5], [[1, -0.3], [-0.3, 1]], 300)
y_popB = (X_popB[:, 0] + X_popB[:, 1] > 10).astype(int)
```

Two separate populations of data are generated here, each containing 300 points with two features. `multivariate_normal` draws points from a two-dimensional Gaussian distribution, where the first argument is the centre of the distribution and the second is a covariance matrix that controls both the spread of each feature and how the two features relate to one another. Population A is centred at [2, 2], with a covariance matrix that gives the two features a positive correlation, meaning that points with a high value on Feature 1 also tend to have a high value on Feature 2. Population B is centred much further away at [5, 5], and its covariance matrix gives the two features a negative correlation instead, so the relationship between the features is reversed compared to Population A. These two populations therefore differ in both location and internal structure, representing two genuinely different distributions.

The class label for each population is then defined by a simple rule based on the sum of its two features. For Population A, a point belongs to class 1 if `X_popA[:, 0] + X_popA[:, 1] > 4`, and class 0 otherwise. For Population B, the same kind of rule is used, but with a different threshold of 10, chosen so that the rule produces a sensible class split given Population B's different centre. The important point is that both populations represent the same underlying task, separating two classes based on a combination of the two features, but the data itself looks very different depending on which population it comes from.

**2. Scaling the Data**

```python
# Scale using Population A's statistics, Population B is DIFFERENT
scaler_iid = StandardScaler()
X_popA_s   = scaler_iid.fit_transform(X_popA)
X_popB_s   = scaler_iid.transform(X_popB)    # same scaler, different distribution
```

A `StandardScaler` is fitted using only Population A, then applied to both populations. This mirrors how scaling works in a real machine learning pipeline: the scaler learns its parameters, the mean and standard deviation of each feature, from the training data, and those same parameters are then used to transform any future data the model encounters, including the test set. Population A's standardised values will therefore be well behaved, with a mean close to zero and a standard deviation close to one, because the scaler was tuned specifically to this data. Population B, however, has a different centre and spread, so applying Population A's scaler to it will not produce the same neatly standardised result, the scaler is, in effect, looking at Population B through a lens calibrated for a different population entirely.

**2. Fitting the Model**

```python
# Train on Population A, evaluate on both
clf_iid = LogisticRegression(random_state=0)
clf_iid.fit(X_popA_s, y_popA)
acc_same_dist = accuracy_score(y_popA, clf_iid.predict(X_popA_s))
acc_diff_dist = accuracy_score(y_popB, clf_iid.predict(X_popB_s))
```

A logistic regression model is trained exclusively on Population A's scaled data and labels. Once trained, this single model is evaluated twice. First, `acc_same_dist` measures its accuracy on Population A, the same distribution it was trained on, which tells us how well the model has learned the task under the conditions it was trained for. Second, `acc_diff_dist` measures the accuracy of that same model, with no retraining or adjustment, on Population B, a distribution it has never encountered. Comparing these two accuracy values is the entire point of the experiment: it isolates the effect of a distribution shift between training and test data, while keeping the model itself, and the underlying task, identical in both cases.

In [ ]:
# Listing 3.
%matplotlib widget
from visualisations.Figure_2 import show
show()

---
The code produces the output above.

On Population A, the same distribution the model was trained on, the logistic regression achieves 99.0% accuracy. This is an excellent result, and if this were the only number available, it would be reasonable to conclude that the model has learned the task very well. The decision boundary it has found closely matches the rule used to generate the labels for Population A, so almost every point is classified correctly.

On Population B, however, accuracy collapses to 43.3%, which is barely better than guessing for a two-class problem. Nothing about the model has changed between these two evaluations; it is exactly the same fitted logistic regression, applied in exactly the same way. What has changed is the data it is being applied to. Population B is centred in a different location, has a different relationship between its two features, and was labelled using a different threshold. The decision boundary the model learned from Population A simply does not align with how Population B is structured, so the model's predictions are largely wrong, despite being produced with complete confidence.

The 55.7 percentage point drop between these two figures is the **cost of the i.i.d. violation**. It did not arise from a flaw in the model, the choice of algorithm, or the amount of training data, it arose purely from the fact that the test data did not come from the same distribution as the training data. This is the practical consequence of the i.i.d. assumption discussed earlier: a test accuracy is only meaningful as an estimate of real-world performance if the test data genuinely represents the conditions the model will face once deployed. If that assumption fails, even a model that performs near perfectly in evaluation can fail almost completely in practice, and the evaluation itself would give no indication that anything was wrong.

---

### 4.3 The No Free Lunch Theorem

The previous two sections looked at problems that arise from how data is collected and split. This final section looks at a different kind of problem, one that concerns the model itself, and it leads to a perhaps surprising conclusion: there is no such thing as a single "best" machine learning algorithm.

A well-known theoretical result, the No Free Lunch (NFL) theorem, states that no single algorithm performs best across all possible problems. If you were to average each algorithm's performance over every conceivable dataset, every algorithm would come out equally well, including approaches as simple as random guessing. An algorithm that is extremely effective on one type of problem must, by this same averaging, perform correspondingly worse on some other type of problem somewhere else in the space of all possible datasets.

This may seem like a purely abstract result, but it has a very practical consequence. It means that statements like "this algorithm is the best" are meaningless without specifying what it is the best *for*. The right choice of model depends entirely on the structure of the specific data you are working with, including how the classes are arranged, how many features there are, and how those features relate to one another. A model that performs extremely well on one dataset can perform poorly on a different dataset, even if both datasets describe superficially similar problems, and there is no shortcut that lets you skip testing multiple models to find out which one suits your particular data.

This is the theoretical justification for an idea that has been implicit throughout this notebook so far: experimental discipline matters because it is the only way to make a fair comparison between models. If different models are evaluated under different conditions, perhaps with different train/test splits, different preprocessing, or different amounts of data, then any difference in their scores could be due to those differing conditions rather than a genuine difference in how well the models suit the problem. Only by holding everything else constant, the same split, the same scaling, the same evaluation metric, can a difference in scores be attributed to the models themselves.

To demonstrate the No Free Lunch theorem directly, we will run six different classifiers on two datasets with deliberately different structures.

1. Dataset 1 is linearly separable, meaning the two classes can be split reasonably well by a single straight line.
2. Dataset 2 is the "two moons" dataset, in which the two classes form interlocking crescent shapes that no straight line can separate, requiring a curved decision boundary instead.

Figure 3 first shows both datasets side by side, so you can see this structural difference directly before any classifiers are introduced. When looking at Figure 3, the key thing to notice is how a straight line could plausibly divide the two classes in the left-hand panel, but no straight line could do the same in the right-hand panel, whatever divides the two crescents in Dataset 2 has to curve.

In [ ]:
# Listing 4.
%matplotlib widget
from visualisations.Figure_3 import show
show()

This structural difference is the entire reason the No Free Lunch theorem matters in practice. The code below trains the same six classifiers on both datasets, using an identical pipeline, the same train/test split ratio, and the same scaling procedure, so the only thing that differs between the two halves of the experiment is the data itself. These classifiers are:

* SVM (linear)
* Random Forest
* Logistic Regression
* k-NN (k=5)
* Decision Tree
* Naive Bayes

Figure 4 plots the per classifier results as a bar chart, with one pair of bars per classifier. Steelblue for Dataset 1 and red for Dataset 2, so the two rankings can be compared directly. Notice how the ranking of the six classifiers changes between the two datasets, sometimes dramatically, even though every other part of the evaluation pipeline is kept identical.

On Dataset 1, the linearly separable data, the linear SVM comes out on top, tied with Random Forest at 96.8%, and logistic regression is not far behind in joint third place. This makes sense given what Figure 3 showed: since a straight line can separate the two classes reasonably well, models that work by finding a linear boundary, such as the linear SVM and logistic regression, are well suited to this data and perform competitively with more flexible models.

On Dataset 2, the two moons data, the picture changes completely. The linear SVM, which came joint first on Dataset 1, drops to joint last place on Dataset 2, and logistic regression, another linear model, is tied with it at the bottom. Meanwhile, Random Forest and k-NN, which rely on flexible, non-linear decision boundaries, move to the top of the ranking. Again, this matches what Figure 3 showed: no straight line can separate the two interlocking crescents well, so the linear models are fundamentally limited here in a way they were not on Dataset 1, while models that can bend their decision boundary to follow the shape of the data perform much better.

Look closely at where each classifier sits in the two rankings, and notice that almost nothing stays in the same position. The linear SVM goes from first to last. Random Forest stays near the top in both, but k-NN moves from joint third to second, and Naive Bayes moves from last to fourth. **No classifier is consistently best or consistently worst across both datasets**, which is exactly the behaviour the **No Free Lunch theorem predicts**: ranking a set of classifiers on one dataset tells you very little about how they will rank on a different dataset, because the ranking depends on how well each classifier's assumptions match the structure of that specific data.

In [ ]:
# Listing 5.
%matplotlib widget
from visualisations.Figure_4 import show
show()

---

## 5. The Experimental Pipeline: Keep Test Conditions Identical

🔙 [Back to Table of Contents](#table-of-contents)

Because the No Free Lunch theorem tells us we must compare multiple models, those comparisons have to be fair. The golden rule is:

> Every model must be trained and evaluated under exactly the same conditions.

This means three things in practice. 

1. First, there is one pre-defined split, the same training data and the same test data are used for every model in the comparison, rather than each model getting its own split.
2. Second, there is one preprocessing pipeline, a scaler is fitted on the training data only, and that same fitted scaler is then applied to the test data, for every model.
3. Third, the test set is never touched during training or model selection, it is used only once, at the very end, as the final referee of how the models compare.

The most common mistake is using a different random seed for each model's split. If Model A's test set happens to contain easier examples than Model B's, Model A will look better in the results, but the difference is just luck, not a genuine difference in how good the models are.

The code below demonstrates the correct approach first. A single dataset is created, and then split once into a training set and a test set:

```python
X_tr_c, X_te_c, y_tr_c, y_te_c = train_test_split(
    X_pipe, y_pipe, test_size=0.25, random_state=42,
)
```

The `random_state=42` here is fixed deliberately. It is not a tuning parameter to experiment with, its only purpose is to ensure that this one split, once created, stays exactly the same for every model evaluated against it. Whatever points end up in `X_te_c` and `y_te_c` are now fixed as "the test set" for the remainder of this experiment.

Next, a scaler is fitted and applied:

```python
sc_c = StandardScaler()
X_tr_c_s = sc_c.fit_transform(X_tr_c)   # fit on training data only
X_te_c_s = sc_c.transform(X_te_c)       # apply the same fitted scaler to test data
```

Next `fit_transform` is used on the training data, this is where the scaler learns the mean and standard deviation of each feature. The test data is then transformed using `transform` alone, not `fit_transform`, so the scaler's parameters are not recalculated based on the test data. The test set is scaled using statistics learned entirely from the training set, exactly as it would be if the test points were genuinely new data arriving after the model had already been built.

Finally, every model in `MODEL_LIST` is trained and evaluated using these same scaled training and test sets:

```python
correct_results = {}
for name, clf in MODEL_LIST:
    clf.fit(X_tr_c_s, y_tr_c)
    correct_results[name] = accuracy_score(y_te_c, clf.predict(X_te_c_s))
```

Each model is fitted on `X_tr_c_s` and `y_tr_c`, and then scored on `X_te_c_s` and `y_te_c`. Because `X_te_c_s` and `y_te_c` are the same for every model in this loop, every accuracy in `correct_results` was measured under identical conditions, so the resulting scores can be compared directly and any difference between them reflects a genuine difference between the models.

Figure 5 puts this correct pattern side by side with the wrong pattern described above. The left panel shows the results from the correct pipeline, and the right panel shows the results from the wrong pipeline, using the same five models and the same dataset in both cases.


In [ ]:
# Listing 6.
%matplotlib widget
from visualisations.Figure_5 import show
show()

The "CORRECT" column shows each model's accuracy when evaluated on the single shared test set, and the "WRONG" column shows each model's accuracy when each was given its own, different split. Notice that every model's score changes, sometimes by less than a percentage point and sometimes, as with Random Forest, by nearly seven percentage points. None of these changes reflect any genuine improvement or weakening in the models themselves, since the models and their settings are identical in both columns. The differences exist only because each model in the "WRONG" column was tested on a different sample of data, some of which happened to be easier or harder than others.

The practical danger here is that a Random Forest's accuracy drops from 94.0% in the correct pipeline to 87.3% in the wrong pipeline, while Decision Tree's accuracy rises slightly from 85.3% to 86.0%. If you only had access to the "WRONG" column, the gap between Random Forest and the other models would look much smaller than it actually is, and you might draw conclusions about which model is best that simply do not hold up once every model is tested under the same conditions. 

This is precisely why the golden rule at the start of this section matters: a single shared test set is what makes the "CORRECT" column a fair comparison, and the "WRONG" column a demonstration of what happens when that rule is broken.



---

## 6. How Much Data for Training vs Testing?

🔙 [Back to Table of Contents](#table-of-contents)

Once we accept that we need a separate test set, the next question is: how big should each part be?

This is a genuine trade-off with no single right answer. More training data generally means the model has more examples to learn from and can usually fit a better decision boundary, but it also leaves fewer samples in the test set, which makes the resulting accuracy estimate noisier and more dependent on which particular samples happened to land in that smaller test set. More test data gives a more stable, reliable estimate of accuracy, but it leaves the model with fewer examples to learn from, which can leave it underpowered, particularly for more complex models or smaller datasets overall.

| More training data | More test data |
|---|---|
| Model can learn more | Model has less to learn from |
| Fewer test samples, noisier estimate | More test samples, more stable estimate |

Common choices are 70/30, 75/25, and 80/20 for train and test respectively. A 90/10 split risks having too few test samples to get a reliable estimate, while a 50/50 split leaves the model with comparatively little data to learn from.

Figure 6 explores this trade-off directly. The same Decision Tree configuration is trained across every split ratio from 10% to 90% training data, and each ratio is repeated 20 times with different random seeds. The left panel plots the mean test accuracy at each split ratio, with a shaded band showing plus or minus one standard deviation across those 20 seeds. A wide band means the accuracy you would measure depends heavily on which particular samples happened to end up in the test set. The right panel plots that same standard deviation directly against the number of samples in the test set, making the relationship between test set size and estimate stability easier to read off.


In [ ]:
# Listing 7.
%matplotlib widget
from visualisations.Figure_6 import show
show()

---

A few things stand out in the results above. The mean accuracy itself barely changes across most of the range, sitting between roughly 0.86 and 0.87 from 50% training data all the way up to 90%, so for this particular dataset and model, having more training data does not translate into a dramatically better model once a reasonable amount of data is already available. What does change more noticeably is the standard deviation, the measure of how much the accuracy estimate bounces around between different random splits. At 80% train / 20% test, the standard deviation is at its lowest, around 0.016, with 200 samples in the test set. At 90% train / 10% test, despite having the most training data of any row, the standard deviation jumps to 0.026, the largest in the table, because the test set has shrunk to only 100 samples, and with so few samples, a handful of easy or hard examples can shift the accuracy noticeably.

This is the practical lesson of Figure 6: pushing the split ratio towards more training data does not necessarily make your evaluation better, and beyond a certain point it can make it worse, because the test set becomes too small to give a stable estimate. The 75/25 and 80/20 results come closest to the best balance for this dataset, with accuracy similar to the higher training percentages but with a smaller standard deviation, which is part of why these ratios are such common defaults in practice.



---

## 7. Class Distributions and Sampling

🔙 [Back to Table of Contents](#table-of-contents)

### 7.1 The Class Imbalance Problem

In many real-world problems, classes are not equally represented. Medical diagnosis datasets might have only 1 to 5% positive cases. Fraud detection datasets might have as little as 0.1% fraudulent transactions. Network intrusion detection might see 99% normal traffic, with genuine intrusions making up the remaining 1%. In each of these cases, one class, sometimes called the minority class, is rare compared to the other, often called the majority class.

When classes are severely imbalanced, accuracy becomes a misleading metric. Consider a disease dataset where 99% of patients are healthy and 1% have the disease. A model that simply predicts "healthy" for every single patient, without looking at any of their data, will be correct 99% of the time, since 99% of patients genuinely are healthy. This gives it an accuracy of 99%, a figure that sounds excellent. Yet this model has caught precisely zero cases of the disease. It has not learned anything about what distinguishes a sick patient from a healthy one, it has simply exploited the fact that healthy patients vastly outnumber sick ones in this dataset. A model this useless should not be rewarded with a 99% accuracy score, but accuracy alone cannot tell the difference between this model and one that has genuinely learned to detect the disease.

This is why understanding and checking your class distribution is always the first step in any evaluation. Before training any model, it is worth asking what accuracy a trivial, "always predict the majority class" strategy would achieve on your data, since this number sets a floor that any genuinely useful model needs to clear, and it tells you immediately whether accuracy alone will be a meaningful metric for this problem or whether other metrics, which we will return to in later notebooks, will be needed instead.

Figure 7 makes this concrete. It computes the majority-class baseline accuracy for several different levels of class imbalance, from a balanced 50/50 split down to a heavily imbalanced 99/1 split, and visualises what these distributions actually look like side by side.

In [ ]:
# Listing 8.
%matplotlib widget
from visualisations.Figure_7 import show
show()

---

At a balanced 50/50 split, always predicting the majority class gets you no further than guessing; an accuracy of 50% tells you essentially nothing useful either way. As the imbalance grows, this baseline climbs steadily, and by the time the minority class makes up only 5% or 1% of the data, a model that has learned absolutely nothing can still score 95% or 99% accuracy simply by ignoring the minority class entirely. The two rows flagged as misleading are the ones where this baseline exceeds 90%, a threshold beyond which an accuracy figure on its own becomes almost meaningless: a genuinely useful model and a model that has learned nothing at all could both report accuracy scores in this range, and accuracy alone gives you no way to tell them apart.


---

### 7.2 Sampling Methods

So far we have talked about how much data goes into the training set versus the test set, but not about how the individual points are chosen for each set. This turns out to matter just as much, particularly on imbalanced data, since two splits with exactly the same train/test ratio can still look very different depending on which points end up where.

Random sampling picks samples for the test set uniformly at random, with no regard for class labels. On a balanced dataset, where each class is roughly equally represented, this tends to work fine, since both classes are common enough that a random sample will usually contain a reasonable number of each. On an imbalanced dataset, however, random sampling gives no guarantee about how many minority class examples end up in the test set. By chance, a particular random split could contain far fewer minority class examples than the true proportion would suggest, or in extreme cases of severe imbalance, none at all, leaving you unable to say anything about how the model performs on that class.

Stratified sampling addresses this directly by sampling from each class separately, in proportion to its size in the full dataset, so that the class balance in the test set matches the class balance in the full dataset as closely as possible. This guarantees the minority class is represented in every split, in roughly its true proportion, regardless of which random seed is used.

In scikit-learn, this is enabled by passing `stratify=y` to `train_test_split`, where `y` is the array of class labels. Without it, a split looks like this:

```python
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=0
)
```

To make this split stratified, the only change needed is to add `stratify=y`:

```python
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=0, stratify=y
)
```

Passing `stratify=y` tells `train_test_split` to look at the class labels in `y` and ensure that the proportion of each class in the resulting training and test sets matches the proportion in the full dataset, rather than leaving this to chance. The split is otherwise performed in exactly the same way as before, with the same `test_size` and `random_state` arguments controlling how much data goes into each set and ensuring the split is reproducible. The only difference is that, with `stratify` set, every split, regardless of the random seed used, will contain approximately the same fraction of minority class examples as the original dataset.

Figure 8 demonstrates the difference between random and stratified sampling directly. A 90/10 imbalanced dataset is split 30 times using each method, and the fraction of the test set belonging to the minority class is recorded each time. The left panel shows how this fraction moves around from repeat to repeat for each method, and the right panel summarises the spread of these values as boxplots, showing numerically how much more stable stratified sampling is.

Random and stratified sampling both aim to produce a test set that looks like a smaller version of the full dataset. A third approach, weighted sampling, takes a different aim entirely: it deliberately changes the class balance of the data used for training, oversampling rare classes or undersampling common ones, so that a model sees more of the minority class than it naturally would. We will look at this approach next, in Section 7.3.

In [ ]:
# Listing 9.
%matplotlib widget
from visualisations.Figure_8 import show
show()

### 7.3 Weighted Sampling: Deliberately Rebalancing

Stratified sampling, covered in the previous section, preserves the original class imbalance in every split: the test set's class balance matches the full dataset's class balance, whatever that happens to be. This is exactly what you want when evaluating a model, since the test set should reflect the real-world conditions the model will eventually face. Training, however, is a different matter. If the minority class makes up only 1% or 5% of the training data, a model trained on that data has very few examples to learn what the minority class actually looks like, and it can often achieve a low training error simply by leaning heavily towards the majority class, without ever learning the minority class's distinguishing features well.

Weighted sampling addresses this by deliberately changing the class balance of the data a model is trained on. Rather than sampling training examples uniformly, or in proportion to their natural frequency, each example is assigned a sampling weight, and minority class examples are given higher weights so that they are selected more often than their natural frequency would suggest. The result is a training set in which the minority class is better represented than it is in the original data, giving the model more opportunities to learn what separates it from the majority class.

It is worth being clear about what weighted sampling changes and what it does not. It changes the data the model is trained on, not the data it is evaluated on; the test set should still reflect the true, real-world class balance, which is exactly why stratified sampling remains the right approach for the test set. Weighted sampling is also not free: oversampling the minority class means some of its examples will appear multiple times in the training data, which can increase the risk of the model overfitting to those specific examples, and undersampling the majority class means discarding some of its data entirely, which can throw away information that would otherwise have been useful.

---

The core idea is simple: make each minority class sample more likely to be picked, so it ends up being used more often when building the new training set.

```python
weights = np.where(y_w == 0, 1 / 900, 1 / 100)
```

This line gives every sample a number representing how likely it is to be chosen. Each class 0 sample gets `1/900`, and each class 1 sample gets `1/100`, nine times larger. Since there are nine times fewer class 1 samples than class 0 samples, giving each one nine times the chance balances things out: if you add up the numbers for all 900 class 0 samples, you get 1, and if you add up the numbers for all 100 class 1 samples, you also get 1. So class 0 as a whole and class 1 as a whole end up equally likely to be picked, even though one group is much bigger than the other.

```python
weights = weights / weights.sum()
```

This line just rescales those numbers so they all add up to exactly 1, turning them into proper probabilities. `rng.choice` needs this to know how likely each sample is to be picked.

```python
sampled_idx = rng_w.choice(len(X_w), size=1000, replace=True, p=weights)
X_balanced = X_w[sampled_idx]
y_balanced = y_w[sampled_idx]
```

Now we pick 1000 samples according to those probabilities. The important detail is `replace=True`, which means a sample can be picked, and then picked again later. Without this, each class 1 sample could only be used once, capping us at 100 minority examples total, the same as before. With repeats allowed, and class 1 samples being so much more likely to be picked each time, the same class 1 points get chosen over and over, so they end up appearing many times in the final 1000 samples. That repetition is exactly how the minority class gets "oversampled". The last two lines use the chosen sample numbers to build `X_balanced` and `y_balanced`, the new, rebalanced training set.

In [ ]:
# Listing 10.
# ── Weighted sampling: deliberately changing the class balance ─────────────
#
# Sometimes we want to change the class proportions deliberately:
#   - oversample the rare class so the model sees more of it
#   - undersample the majority class to reduce training time
#
# This code demonstrates weighted sampling using NumPy's rng.choice with
# custom per-sample weights, then compares per-class accuracy when training
# on the original imbalanced data versus the rebalanced sample.
#
# Note: scikit-learn classifiers also accept a class_weight parameter, which
# achieves a similar effect without resampling.
rng_w = np.random.default_rng(5)

# Construct a 90/10 imbalanced dataset with separable class means, so that
# any change in per-class accuracy can be attributed to the class balance
# of the training data, not to the classes being inherently hard to tell apart.
X_w0 = rng_w.normal([1, 1], 1.0, (900, 2))   # majority class, centred at (1, 1)
X_w1 = rng_w.normal([3, 3], 1.0, (100, 2))   # minority class, centred at (3, 3)
X_w = np.vstack([X_w0, X_w1])
y_w = np.array([0] * 900 + [1] * 100)

# Assign each sample a weight inversely proportional to the size of its
# class. With 900 class 0 samples and 100 class 1 samples, each individual
# class 1 sample receives 9 times the weight of a class 0 sample, so that,
# on average, both classes are equally likely to be drawn regardless of
# their original counts.
weights = np.where(y_w == 0, 1 / 900, 1 / 100)   # inverse frequency weighting
weights = weights / weights.sum()                 # normalise so weights sum to 1, as required by rng.choice's p argument

# Draw 1000 samples with replacement according to these weights. Sampling
# WITH replacement is what allows the minority class to be "oversampled",
# the same minority points can, and will, be selected more than once.
sampled_idx = rng_w.choice(len(X_w), size=1000, replace=True, p=weights)
X_balanced = X_w[sampled_idx]
y_balanced = y_w[sampled_idx]

print('After weighted sampling:')
print(f'  Class 0: {(y_balanced == 0).sum()} ({(y_balanced == 0).mean():.1%})')
print(f'  Class 1: {(y_balanced == 1).sum()} ({(y_balanced == 1).mean():.1%})')
print()

# A single stratified test set, drawn from the ORIGINAL data, is used to
# evaluate both training approaches. Keeping this test set fixed and
# representative of the true class balance is what makes the comparison
# below fair, following the same pattern as Section 5.
X_tr_w2, X_te_w2, y_tr_w2, y_te_w2 = train_test_split(
    X_w, y_w, test_size=0.25, random_state=0, stratify=y_w
)

for name, X_tr_use, y_tr_use in [
    ('Original (90/10)',    X_tr_w2,    y_tr_w2),
    ('Weighted (balanced)', X_balanced, y_balanced),
]:
    # A fresh scaler is fitted for each training set, since the weighted
    # sample has a different distribution of points to the original
    # training set, and each model should be scaled using the statistics
    # of the data it was actually trained on.
    sc_w2 = StandardScaler()
    clf_w = DecisionTreeClassifier(max_depth=5, random_state=0)
    clf_w.fit(sc_w2.fit_transform(X_tr_use), y_tr_use)
    y_pred_w = clf_w.predict(sc_w2.transform(X_te_w2))

    # Accuracy is reported separately for each class, as well as overall,
    # since overall accuracy on a 90/10 test set can stay high even if
    # class 1 accuracy is poor, exactly the problem raised in Section 7.1.
    class0_acc = accuracy_score(y_te_w2[y_te_w2 == 0], y_pred_w[y_te_w2 == 0])
    class1_acc = accuracy_score(y_te_w2[y_te_w2 == 1], y_pred_w[y_te_w2 == 1])
    overall = accuracy_score(y_te_w2, y_pred_w)

    print(f'{name}:')
    print(f'  Class 0 accuracy: {class0_acc:.1%}')
    print(f'  Class 1 accuracy: {class1_acc:.1%}')
    print(f'  Overall accuracy: {overall:.1%}')
    print()

Both oversampling the minority class and undersampling the majority class come with trade-offs that are worth understanding before using either technique.

Oversampling the minority class, as in the weighted sampling approach above, works by repeating existing minority class examples until the classes are balanced. The risk here is that the model can start to memorise these specific, repeated examples rather than learning the general pattern that separates the two classes. If a handful of minority class points appear dozens of times in the training set, the model has effectively been shown the same few examples over and over, and it may become overly confident about the exact characteristics of those particular points, characteristics that might just be noise, rather than learning what makes the minority class distinct in general. This is a form of overfitting specific to the minority class, and it can be especially problematic when the minority class has very few unique examples to begin with.

Undersampling the majority class takes the opposite approach: rather than repeating minority class examples, some majority class examples are simply discarded so that the two classes are closer in size. The risk here is more straightforward: discarding data throws away information. If the majority class contains useful variety, different sub-groups, edge cases, or patterns, that variety can be lost when a large portion of the majority class is removed, potentially leaving the model with an incomplete picture of what the majority class looks like. This can be particularly costly when the dataset is small to begin with, since undersampling shrinks the total amount of training data available even further.

In practice, both techniques are tools rather than fixes, and whether they help depends on the specific dataset and problem. The comparison in the code above, between a model trained on the original imbalanced data and one trained on a weighted, rebalanced sample, is one way to check empirically whether rebalancing actually improves performance on the minority class for your particular data, rather than assuming it always will.

---

## 8. K-Fold Cross Validation

🔙 [Back to Table of Contents](#table-of-contents)

A single train/test split, the approach used throughout this notebook so far, has a fundamental weakness: the result depends on which particular samples happened to end up in the test set. If, by chance, the test set contains harder-than-average examples, accuracy looks worse than the model's true performance suggests, and if it contains easier-than-average examples, accuracy looks correspondingly better. Either way, a single number from a single split is, in part, a reflection of luck.

K-fold cross-validation addresses this by ensuring every data point serves as a test sample exactly once, rather than relying on a single, fixed test set. The procedure works as follows. First, the data is divided into $k$ equally sized groups, called folds, with stratification used to preserve the original class proportions within each fold, following the same idea introduced in Section 7.2. Then, for each fold $i$ from 1 to $k$ in turn, the model is trained on all the folds except fold $i$, and evaluated on fold $i$ alone, with the resulting accuracy recorded. Once all $k$ folds have each played the role of test fold exactly once, the $k$ recorded accuracy scores are summarised in two ways: their mean, often called the CV accuracy, and their standard deviation.

The mean Cross-Validation (CV) accuracy is a far more reliable estimate of how the model is likely to perform than a single split's accuracy, because it is effectively an average over $k$ different test sets rather than just one, smoothing out the effect of any single set of samples being unusually easy or hard. The standard deviation adds something a single split cannot give you at all: a sense of how sensitive the model's performance is to which data it happens to see. A small standard deviation means the model performed similarly regardless of which fold was held out; a large standard deviation means performance varied considerably from fold to fold, which is itself useful information about how stable the model is.

Figure 9 visualises what 5-fold cross-validation looks like in practice. Each row represents one iteration of the procedure, with the data divided into the same 5 folds in every row. The tomato block in each row shows which fold is acting as the test fold for that iteration, while the remaining blocks show the folds used for training. Reading down the rows, notice that the tomato block moves to a different fold each time, so that across all 5 rows, every fold takes its turn as the test fold exactly once, and is used for training in every other row.

In [ ]:
# Listing 11.
%matplotlib widget
from visualisations.Figure_9 import show
show()

The code below implements k-fold cross-validation using scikit-learn's `StratifiedKFold` and `cross_val_score`, which together handle the bookkeeping shown in Figure 9, dividing the data into folds, rotating which fold acts as the test set, and preserving class proportions within each fold, all automatically.

`StratifiedKFold` is found in `sklearn.model_selection`, alongside `train_test_split`, and is created like this:

```python
from sklearn.model_selection import StratifiedKFold

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=0)
```

The `n_splits` parameter is $k$, the number of folds the data will be divided into; 5 and 10 are the most common choices, and we will return to why in the next section. `shuffle=True` tells `StratifiedKFold` to shuffle the data before splitting it into folds. This matters if your data happens to be ordered in some way, for example sorted by date or grouped by source, since splitting unshuffled, ordered data into folds could otherwise produce folds that are systematically different from one another, echoing the i.i.d. concerns raised in Section 4.2. `random_state=0` fixes that shuffling so the folds are reproducible.

Creating a `StratifiedKFold` instance like this does not, by itself, split any data. It is simply a configuration object that describes how the splitting should be done: into how many folds, whether to shuffle, and with what random seed. This configuration is exactly what the diagram in Figure 9 represents: $k$ folds, each one taking a turn as the test fold while the remaining folds form the training set, with the class balance in each fold matching the original data.

To actually run cross-validation using this configuration, the `skf` object is passed into scikit-learn's `cross_val_score`, a higher-level function that carries out the entire procedure shown in Figure 9 in one call. For each fold, it fits a fresh copy of the given model on that fold's training portion, scores it on that fold's test portion, and collects all of the resulting scores into a single array. The code below shows this in full, including the dataset setup and scaling that come before the cross-validation step itself.

In [ ]:
# Listing 12.
# This snippet uses sklearn's StratifiedKFold and cross_val_score, the
# higher-level scikit-learn approach described above. It assumes a manual,
# step-by-step k-fold loop has already been shown earlier in this notebook,
# so this code can be compared directly against that manual version.

# A fresh dataset is generated for this comparison, separate from any used
# earlier in the notebook, so the cross-validation results here are not
# affected by anything done previously.
X_kf, y_kf = make_classification(
    n_samples=500, n_features=8, n_informative=4, random_state=5
)

# The data is scaled once, up front, across the whole dataset, purely to
# keep this example short and focused on cross-validation itself. This is
# a simplification: scaling the full dataset before splitting it into folds
# means each fold's scaling is influenced very slightly by the other folds,
# a small amount of data leakage. In a production pipeline, this is avoided
# by wrapping the scaler and model together in a Pipeline, so that scaling
# is fitted fresh inside each fold, using only that fold's training data.
sc_kf2 = StandardScaler()
X_kf_s = sc_kf2.fit_transform(X_kf)

# cross_val_score automates the entire k-fold loop: for each fold it fits a
# fresh copy of the given model on that fold's training portion, scores it
# on that fold's test portion, and collects the results.
#
#   - First argument: the model to evaluate. A fresh, untrained instance is
#     passed in, cross_val_score clones it for each fold rather than reusing
#     a single fitted model.
#   - X_kf_s, y_kf: the full (scaled) feature matrix and labels. The
#     splitting into folds is handled internally, you do not pre-split
#     the data yourself.
#   - cv: controls how the folds are created. Passing a StratifiedKFold
#     instance, rather than just an integer, lets us specify shuffling and
#     a random seed, and ensures each fold preserves the original class
#     proportions, following the same pattern introduced in Section 7.2.
#       - n_splits=5: k = 5 folds, matching Figure 9.
#       - shuffle=True: shuffles the data before splitting into folds, so
#         the fold order does not reflect any ordering in the original data.
#       - random_state=0: fixes that shuffle, so the folds, and therefore
#         the results, are reproducible.
#   - scoring='accuracy': the metric calculated on each fold's test portion.
#     cross_val_score supports many other metrics via this argument, which
#     we will return to in later sections.
cv_scores = cross_val_score(
    DecisionTreeClassifier(max_depth=5, random_state=0),
    X_kf_s, y_kf,
    cv=StratifiedKFold(n_splits=5, shuffle=True, random_state=0),
    scoring='accuracy'
)

# cv_scores is a NumPy array containing one accuracy value per fold, in the
# order the folds were evaluated. Printing each fold's score individually,
# alongside the mean and standard deviation, mirrors the manual k-fold
# output above, so the two approaches can be compared directly.
print('sklearn StratifiedKFold 5-fold cross-validation:')
for i, acc in enumerate(cv_scores, 1):
    print(f'  Fold {i}: {acc:.4f}')
print(f'  Mean: {cv_scores.mean():.4f}')
print(f'  Std:  {cv_scores.std():.4f}')

---

### 8.1 K-Fold vs Single Split: Stability

The previous section explained, in principle, why k-fold cross-validation should give a more reliable accuracy estimate than a single train/test split, because it averages over $k$ different test folds rather than relying on just one. Figure 10 puts this claim to the test directly, by running both approaches many times over and comparing how much their results vary.

The same dataset and model are evaluated 50 times using each approach: 50 times with a single train/test split, using a different random seed each time, and 50 times with 5-fold cross-validation, also with a different seed each time, where each cross-validation run produces one estimate by averaging its own 5 folds. This gives two sets of 50 accuracy estimates, one for each approach, and the figure compares how spread out these two sets are.

The left panel shows these two sets of estimates as overlapping histograms. The tomato histogram, for single splits, is noticeably wider and flatter, spread across a broader range of accuracy values. The steelblue histogram, for 5-fold cross-validation, is narrower and more sharply peaked around its mean, shown by the dashed steelblue line. Both distributions are centred in roughly the same place, since both approaches are estimating the same underlying quantity, but the cross-validation estimates cluster much more tightly around that centre.

The right panel shows the same 100 numbers differently, plotting each repetition's accuracy estimate in sequence. The tomato line, for single splits, jumps around noticeably from one seed to the next, while the steelblue line, for cross-validation, stays much closer to a flat, stable level throughout. A single train/test split's accuracy is, in part, a reflection of which particular samples happened to land in that one test set, and this panel makes that dependence on luck visible as run-to-run noise. Cross-validation's averaging over 5 folds smooths most of that noise out, so its accuracy estimate barely moves even as the random seed changes.

The printed summary beneath the figure puts a number on this difference: the standard deviation of the single-split estimates is several times larger than the standard deviation of the cross-validation estimates, even though their means are very close. This is the practical payoff of cross-validation: for the same underlying model and data, it gives an estimate that is far less sensitive to the particular split used, making it a far more trustworthy figure to report or to use when comparing models.

In [ ]:
# Listing 13.
%matplotlib widget
from visualisations.Figure_10 import show
show()

---

### 8.2 Choosing k: the Bias-Variance Trade-Off

Section 8 introduced $k$-fold cross-validation as a way of getting a more reliable accuracy estimate than a single split, and Figure 10 showed that 5-fold CV is indeed more stable than a single split. But $k$ itself was simply fixed at 5 without explaining why, and the value of $k$ turns out to be a choice with its own trade-off, between how biased the estimate is and how much it varies from fold to fold.

With a small $k$, such as $k = 2$, the data is divided into only two folds, so each training fold contains only about half the data. The model is trained on this smaller amount of data, so it tends to be somewhat undertrained compared to a model trained on more data, and as a result its accuracy on the held-out fold tends to be a little lower than the model's true potential. The CV estimate is therefore biased downward; it is somewhat pessimistic. On the other hand, each test fold is large, around half the data, so the accuracy measured on it is based on a substantial number of samples, giving a relatively stable, low-variance estimate.

With a large $k$, such as $k = 10$ or even $k = n$ (one fold per sample), each training fold contains nearly all of the data, so the model is trained on close to the full dataset and tends to reach close to its true potential. The CV estimate therefore has low bias. However, each test fold is now very small, in the extreme case of $k = n$, just a single sample, so the accuracy measured on any one fold is highly sensitive to whether that one fold happened to contain easy or hard examples. This gives a higher-variance estimate, as also seen directly in Figure 11.

$k = 5$ or $k = 10$ has become the standard empirical compromise between these two effects: training folds are large enough that the model is close to well-trained, while test folds are still large enough to give a reasonably stable per-fold accuracy. The extreme case $k = n$, where every fold contains exactly one sample, is known as leave-one-out cross-validation, or LOO. LOO has very low bias, since the model is trained on all but one sample each time, but it requires training $n$ separate models, which becomes computationally expensive for anything beyond small datasets, and as just described, its individual fold scores can be highly variable.

Figure 11 makes this trade-off concrete by running cross-validation with $k$ values of 2, 3, 5, 10, and 20 on the same dataset and model. The left panel shows the mean CV accuracy for each $k$, with error bars showing plus or minus one standard deviation, while the right panel isolates the standard deviation itself as a bar chart. Looking at the left panel, notice how the mean accuracy shifts slightly as $k$ increases, reflecting the reduction in bias as training folds grow larger. Looking at the right panel, notice how the standard deviation tends to grow as $k$ increases, reflecting the corresponding increase in variance as test folds shrink. Together, these two panels show why $k = 5$ or $k = 10$ sit in a comfortable middle ground, far enough from $k = 2$ that bias is low, and far enough from $k = 20$ or beyond that variance has not yet grown large.

In virtually all practical situations, $k = 5$ or $k = 10$ with stratification, as covered in Section 8.1, is the right choice, and this is the default you should reach for unless you have a specific reason to consider something else.

In [ ]:
# Listing 14.
%matplotlib widget
from visualisations.Figure_11 import show
show()

---

## 9. The Confusion Matrix

🔙 [Back to Table of Contents](#table-of-contents)

Accuracy gives one number. But a single number cannot tell you what kind of mistakes a model makes, and in many applications the type of mistake matters enormously. Missing a genuine case of disease in a medical screening test is a very different kind of error from incorrectly flagging a healthy person as needing further tests, even though both would simply count as "one wrong prediction" if all you looked at was overall accuracy.

The confusion matrix addresses this by unpacking a model's predictions into a table that shows, for each true class, how many examples were predicted into each possible class. Rather than collapsing everything down to a single percentage, it shows exactly where the model's correct and incorrect predictions fall. Accuracy, in fact, is simply the fraction of entries that fall on the diagonal of this table, the cases where the predicted class matches the true class.

### 9.1 Binary Confusion Matrix

For a two-class problem, with classes conventionally labelled "positive" and "negative", the confusion matrix has four cells:

| | Predicted Positive | Predicted Negative |
|---|---|---|
| **Actual Positive** | True Positive (TP) | False Negative (FN) |
| **Actual Negative** | False Positive (FP) | True Negative (TN) |

Each cell counts a different combination of what the true class actually was and what the model predicted. A true positive is a case that was genuinely positive and was correctly predicted as positive. A true negative is a case that was genuinely negative and was correctly predicted as negative. These two cells, TP and TN, sit on the diagonal and represent correct predictions. A false negative is a case that was genuinely positive but was incorrectly predicted as negative: the model missed it. A false positive is a case that was genuinely negative but was incorrectly predicted as positive: the model raised a false alarm. These two cells, FN and FP, are the two distinct types of error a binary classifier can make.

Accuracy can be written directly in terms of these four counts:

$$
\text{Accuracy} = \frac{TP + TN}{TP + TN + FP + FN}
$$

The numerator is the sum of the two correct-prediction cells, and the denominator is the total number of predictions made. This is exactly the diagonal-fraction idea described above, written out for the binary case.

Which type of error, FN or FP, matters more depends entirely on the application, and a single accuracy figure cannot distinguish between a model that makes mostly false negatives and one that makes mostly false positives, even if both have identical accuracy. Notebook 11 will introduce precision, recall, and related metrics that weight these two error types differently, giving you a way to express which kind of mistake you care about most. For now, the confusion matrix itself is the tool for seeing where errors occur, and Figure 12 gives you a way to explore it directly: an interactive confusion matrix where you can adjust the values of TP, FN, FP, and TN with sliders, and immediately see how accuracy, along with precision and recall as a preview of what is to come in Notebook 11, responds to changes in each cell.

In [ ]:
# Listing 15.
%matplotlib widget
from visualisations.Figure_12 import show
show()

---

### 9.2 Binary Confusion Matrix with Seaborn

Plotting a confusion matrix as a heatmap is something you will likely want to do often, so it is worth understanding each part of a call like the one below, since the same pattern can be reused for any confusion matrix, of any size.

Seaborn is a plotting library built on top of Matplotlib, designed to make certain kinds of statistical plots, including heatmaps, quicker to produce and better looking by default. It is conventionally imported as `sns`:

```python
import seaborn as sns
```

A heatmap is simply a grid of coloured cells, where the colour of each cell represents its value: larger values get one colour, smaller values get another, with a gradient in between. This makes it a natural way to display a confusion matrix, since a confusion matrix is itself just a grid of numbers, and colouring each cell by its count gives an immediate visual sense of where most of the predictions fall, in addition to the exact numbers themselves.

```python
sns.heatmap(
    cm, annot=True, fmt='d', cmap='Blues',
    xticklabels=['Pred. Class A', 'Pred. Class B'],
    yticklabels=['Actual Class A', 'Actual Class B'],
    linewidths=0.5, linecolor='white', ax=ax, annot_kws={'size': 14}
)
ax.set_title('Confusion matrix')
ax.set_xlabel('Predicted label')
ax.set_ylabel('True label')
```

The first argument, `cm`, is the confusion matrix itself, a 2D array where each entry is a count, with rows representing the true class and columns representing the predicted class. This is the only required argument; everything else controls how that array is displayed.

`annot=True` tells seaborn to write each cell's numeric value directly onto the heatmap, on top of its coloured background. Without this, you would see only the colours, which give a rough visual impression of which cells are largest, but not the exact counts. `fmt='d'` controls how those numbers are formatted: `d` means "format as an integer", which is appropriate here since confusion matrix entries are always whole numbers of samples. If you were displaying a matrix of proportions instead of counts, you would need a different format such as `'.1%'` or `'.2f'`.

`cmap='Blues'` sets the colour scheme used to shade each cell according to its value: here, larger counts are shown in darker blue, and smaller counts in lighter blue. Seaborn supports many named colour maps, for example `'YlOrRd'` shades from yellow through orange to red, and the choice is purely about visual style; it does not change the underlying numbers.

`xticklabels` and `yticklabels` set the labels along the bottom and left side of the heatmap respectively. These should describe what the columns and rows represent, following the convention that columns are the predicted class and rows are the true class. The order of the labels matters: the first label corresponds to the first row or column of the matrix, the second label to the second row or column, and so on, so these need to match the order of the classes as they appear in your confusion matrix.

`linewidths=0.5` and `linecolor='white'` add a thin white line between each cell, which makes the grid of cells easier to read at a glance, particularly when adjacent cells have similar colours. `ax=ax` tells seaborn which matplotlib axes to draw onto; this is what allows the heatmap to sit inside a larger figure alongside other panels rather than taking up an entire figure on its own. `annot_kws={'size': 14}` is a dictionary of keyword arguments passed through to the text used for the `annot=True` labels, here, setting their font size to 14 so the numbers are large enough to read clearly.

After calling `sns.heatmap`, the remaining lines are ordinary matplotlib calls: setting a title for the plot, and labelling the x-axis as "Predicted label" and the y-axis as "True label", following the standard confusion matrix convention.

To reuse this pattern for your own confusion matrix, the things you typically need to change are the matrix itself, and the `xticklabels`/`yticklabels`, to match the names and order of your own classes, everything else can usually stay as it is.

---

The code below puts the ideas from Section 9.1 and 9.2 into practice on a real dataset, the breast cancer dataset that scikit-learn provides built in via `load_breast_cancer()`. The dataset itself is not the focus here; it is simply a convenient, ready-to-use binary classification problem (label 0 = malignant, label 1 = benign, I've shown you it before) that lets us build and inspect a confusion matrix without needing to generate or load any data of our own. Everything shown here applies equally to any binary classification problem you might work with.

The first part of the code follows the same pattern used throughout this notebook: load the features and labels into `X_bc` and `y_bc`, create a single train/test split with stratification, fit a scaler on the training data only, and train a classifier, here, logistic regression, on the scaled training data. This part produces nothing new conceptually; its job is simply to produce a set of predictions, `y_pred_bc`, that we can then compare against the true labels, `y_te_bc`.

The interesting part, for the purposes of this section, is what happens next: turning those predictions and true labels into the four confusion matrix counts. This is done with four lines, one per cell of the confusion matrix:

```python
TP = int(np.sum((y_pred_bc == 1) & (y_te_bc == 1)))
TN = int(np.sum((y_pred_bc == 0) & (y_te_bc == 0)))
FP = int(np.sum((y_pred_bc == 1) & (y_te_bc == 0)))
FN = int(np.sum((y_pred_bc == 0) & (y_te_bc == 1)))
```

Each line follows the same recipe. `y_pred_bc == 1` is an array of `True`/`False` values, one per test sample, indicating whether the model predicted class 1 for that sample. `y_te_bc == 1` is a similar array, but for the true label. The `&` operator combines these two arrays element-wise, giving `True` only where both conditions hold for a given sample, for example, both "predicted 1" and "actually 1" for the TP line. `np.sum` then counts how many `True` values there are, since `True` and `False` behave as 1 and 0 when summed, and `int(...)` converts the result from a NumPy integer type to a plain Python integer for cleaner printing later. The only thing that changes from line to line is which two conditions are combined, matching the four cells of the table in Section 9.1: TP is "predicted 1 and actually 1", TN is "predicted 0 and actually 0", FP is "predicted 1 but actually 0", and FN is "predicted 0 but actually 1".

This pattern, comparing `y_pred == k` and `y_true == k` for each combination of predicted and true labels and counting how often both are true, is the general recipe for building a confusion matrix from scratch for any binary classifier, and it extends naturally to more than two classes by considering every combination of predicted and true class, as we will see in Section 9.3.

Once you have TP, TN, FP, and FN, computing accuracy is just the formula from Section 9.1 applied directly: `(TP + TN) / (TP + TN + FP + FN)`. The code prints this manual calculation alongside scikit-learn's own `accuracy_score(y_te_bc, y_pred_bc)`, so you can confirm the two agree: your manual count and scikit-learn's built-in function are computing exactly the same thing.

For visualisation, the four counts are assembled into a 2x2 array, `cm_bc`, arranged so that rows correspond to the actual class and columns correspond to the predicted class, the same layout used by scikit-learn's own `confusion_matrix` function when called as `confusion_matrix(y_true, y_pred, labels=[0, 1])`. In fact, if you would rather not build the counts up manually as shown above, this single call:

```python
from sklearn.metrics import confusion_matrix
cm_bc = confusion_matrix(y_te_bc, y_pred_bc)
```

gives you the same 2x2 array directly. The manual approach above is shown first because it makes explicit exactly what each cell means and how it is derived from your predictions and true labels, but in your own work, `confusion_matrix` is the quicker route to the same result once that meaning is clear.

The remainder of the code turns `cm_bc`, or equivalently the four counts TP, TN, FP, and FN, into Figure 13: a heatmap on the left, produced with `seaborn`'s `sns.heatmap`, and a colour-coded cell diagram on the right, built from individual rectangles and text labels, which spells out what each cell represents in words. Both panels are driven entirely by the same four numbers, so if you substitute in TP, TN, FP, and FN from your own classifier and data, both panels will update accordingly.

In [ ]:
# Listing 16.
# ── The confusion matrix — binary classification ─────────────────────────
#
# Accuracy alone does not tell the full story. The confusion matrix unpacks
# accuracy into four counts:
#
#   TRUE POSITIVE  (TP): predicted positive, actually positive,  correct
#   TRUE NEGATIVE  (TN): predicted negative, actually negative,  correct
#   FALSE POSITIVE (FP): predicted positive, actually negative,  wrong (Type I error)
#   FALSE NEGATIVE (FN): predicted negative, actually positive,  wrong (Type II error)
#
# The matrix is laid out as:
#
#               Predicted Positive    Predicted Negative
#   Actual Pos         TP                   FN
#   Actual Neg         FP                   TN
#
# Accuracy = (TP + TN) / (TP + TN + FP + FN)

# Breast cancer dataset: label 0 = malignant, label 1 = benign
raw_bc = load_breast_cancer()
X_bc, y_bc = raw_bc.data, raw_bc.target

X_tr_bc, X_te_bc, y_tr_bc, y_te_bc = train_test_split(
    X_bc, y_bc, test_size=0.25, random_state=0, stratify=y_bc
)

sc_bc = StandardScaler()
X_tr_bc_s = sc_bc.fit_transform(X_tr_bc)   # fit on training data only
X_te_bc_s = sc_bc.transform(X_te_bc)       # apply same fitted scaler to test data

clf_bc = LogisticRegression(max_iter=1000, random_state=0)
clf_bc.fit(X_tr_bc_s, y_tr_bc)
y_pred_bc = clf_bc.predict(X_te_bc_s)

# Build the confusion matrix components manually, by counting how many test
# predictions fall into each of the four TP / TN / FP / FN categories.
TP = int(np.sum((y_pred_bc == 1) & (y_te_bc == 1)))   # predicted benign, actually benign
TN = int(np.sum((y_pred_bc == 0) & (y_te_bc == 0)))   # predicted malignant, actually malignant
FP = int(np.sum((y_pred_bc == 1) & (y_te_bc == 0)))   # predicted benign, actually malignant
FN = int(np.sum((y_pred_bc == 0) & (y_te_bc == 1)))   # predicted malignant, actually benign

acc_manual = (TP + TN) / (TP + TN + FP + FN)

print('Confusion matrix (manual counts):')
print(f'                     Predicted Malignant  Predicted Benign')
print(f'  Actual Malignant:       {TN:<18} {FP}')
print(f'  Actual Benign:          {FN:<18} {TP}')
print()
print(f'  TP = {TP}  (correctly predicted Benign)')
print(f'  TN = {TN}  (correctly predicted Malignant)')
print(f'  FP = {FP}   (predicted Benign, actually Malignant)')
print(f'  FN = {FN}   (predicted Malignant, actually Benign)')
print()
print(f'  Accuracy = (TP + TN) / (TP + TN + FP + FN)')
print(f'           = ({TP} + {TN}) / ({TP + TN + FP + FN})')
print(f'           = {acc_manual:.4f}')
print(f'  sklearn accuracy_score: {accuracy_score(y_te_bc, y_pred_bc):.4f}')

# ── Visualise: heatmap + colour-coded cell diagram ─────────────────────────
#
# cm follows the same row/column convention as sklearn's confusion_matrix
# with labels=[0, 1]: rows are the actual class, columns are the predicted
# class.
#   cm[0, 0] = TN  (actual 0, predicted 0)
#   cm[0, 1] = FP  (actual 0, predicted 1)
#   cm[1, 0] = FN  (actual 1, predicted 0)
#   cm[1, 1] = TP  (actual 1, predicted 1)
cm_bc = np.array([[TN, FP], [FN, TP]])

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

fig.canvas.header_visible = False
fig.canvas.toolbar_visible = True
fig.canvas.toolbar_position = 'right'

# Left panel: annotated heatmap of the confusion matrix counts.
ax = axes[0]
sns.heatmap(
    cm_bc, annot=True, fmt='d', cmap='Blues',
    xticklabels=['Pred. Malignant', 'Pred. Benign'],
    yticklabels=['Actual Malignant', 'Actual Benign'],
    linewidths=0.5, linecolor='white', ax=ax, annot_kws={'size': 14}
)
ax.set_title(f'Confusion matrix\naccuracy = {acc_manual:.1%}')
ax.set_xlabel('Predicted label')
ax.set_ylabel('True label')

# Right panel: a colour-coded cell diagram naming each cell and marking it
# as correct (steelblue) or wrong (tomato), so the TP/TN/FP/FN labels can
# be tied directly to the heatmap's layout.
ax = axes[1]
cell_colours = [['steelblue', 'tomato'],
                 ['tomato',    'steelblue']]
cell_labels  = [[f'TN = {TN}\n(correct)', f'FP = {FP}\n(wrong)'],
                [f'FN = {FN}\n(wrong)',   f'TP = {TP}\n(correct)']]

for i in range(2):
    for j in range(2):
        rect = plt.Rectangle((j, 1 - i), 1, 1, color=cell_colours[i][j], alpha=0.75)
        ax.add_patch(rect)
        ax.text(j + 0.5, 1 - i + 0.5, cell_labels[i][j],
                ha='center', va='center', fontsize=11, color='white', fontweight='bold')

ax.set_xlim(0, 2)
ax.set_ylim(0, 2)
ax.set_xticks([0.5, 1.5])
ax.set_xticklabels(['Pred. Malignant', 'Pred. Benign'])
ax.set_yticks([0.5, 1.5])
ax.set_yticklabels(['Actual Benign', 'Actual Malignant'])
ax.set_title('TP / TN / FP / FN explained\nblue = correct, tomato = wrong')
ax.set_aspect('equal')

plt.suptitle(
    'Figure 13: confusion matrix, breaking accuracy into its components',
    fontsize=12, y=0.98
)
plt.tight_layout()
plt.show()

### 9.3 Multi-Class Confusion Matrix

Everything covered so far has used a binary confusion matrix, with exactly two classes and four cells. Most of the ideas carry over directly to problems with more than two classes, but the matrix itself grows: for a problem with $n$ classes, the confusion matrix becomes an $n \times n$ grid, with one row and one column for each class. As before, rows represent the actual class and columns represent the predicted class. The diagonal entries are correct predictions: the count of test samples where the predicted class matched the actual class. Every off-diagonal entry represents a misclassification, and crucially, its position in the grid tells you which two classes were confused with each other; the row gives the true class and the column gives the class the model predicted instead.

This additional information is invaluable, and it is exactly what the binary confusion matrix's two off-diagonal cells (FP and FN) cannot provide once there are more than two classes to mix up. If Class A is frequently confused with Class B but never with Class C, that tells you something specific about the data's structure: Class A and Class B likely look similar to the model in feature space, while Class C is more distinct. This kind of pattern can suggest which features might help separate the commonly confused classes, or whether two classes are perhaps not as distinct as their labels imply.

Accuracy is calculated in exactly the same way as in the binary case, just generalised to a grid of any size:

$$
\text{Accuracy} = \frac{\text{sum of diagonal}}{\text{sum of all entries}}
$$

The sum of the diagonal counts every correct prediction, regardless of which class it belongs to, and the sum of all entries is simply the total number of predictions made, so this ratio is still "fraction correct", exactly as before.

The code below makes this concrete using the Iris dataset (again I've shown you this before), which has three classes. Two decision trees are trained on the same data, one with `max_depth=4`, deep enough to learn a reasonably good decision boundary, and one with `max_depth=1`, deliberately too shallow to do the task well. Figure 14 shows both classifiers' $3 \times 3$ confusion matrices side by side, so you can see not only that the shallow tree is less accurate overall, but specifically which classes it confuses that the deeper tree does not, and the code following the figure verifies each classifier's accuracy by summing the diagonal of its confusion matrix directly.

In [ ]:
# Listing 17.
# ── The confusion matrix — multi-class problems ───────────────────────────
#
# For multi-class problems, the confusion matrix becomes an n x n grid.
#
# Layout:
#   rows    = actual class
#   columns = predicted class
#   diagonal cells     = correct predictions
#   off-diagonal cells = misclassifications, identifying which classes
#                        are confused with which
#
# Accuracy = sum(diagonal) / sum(all cells)
#
# The off-diagonal pattern can reveal structure in the data, for example,
# if Setosa is never confused with Virginica but is sometimes confused with
# Versicolor, that tells you Setosa and Versicolor are more similar in
# feature space than Setosa and Virginica are.

iris_data = load_iris()
X_ir, y_ir = iris_data.data, iris_data.target
class_names_ir = iris_data.target_names

X_tr_ir, X_te_ir, y_tr_ir, y_te_ir = train_test_split(
    X_ir, y_ir, test_size=0.30, random_state=0, stratify=y_ir
)

sc_ir = StandardScaler()
X_tr_ir_s = sc_ir.fit_transform(X_tr_ir)   # fit on training data only
X_te_ir_s = sc_ir.transform(X_te_ir)       # apply same fitted scaler to test data

# Compare a well-tuned tree (depth=4) against a deliberately shallow tree
# (depth=1), so the confusion matrices below show both a strong and a weak
# classifier on the same data.
classifiers_ir = [
    ('Decision Tree (good, depth=4)', DecisionTreeClassifier(max_depth=4, random_state=0)),
    ('Decision Tree (poor, depth=1)', DecisionTreeClassifier(max_depth=1, random_state=0)),
]

# Fit each classifier once, and store its predictions and confusion matrix
# for reuse in both the plot and the manual accuracy check below, rather
# than predicting twice.
results_ir = []
for name, clf in classifiers_ir:
    clf.fit(X_tr_ir_s, y_tr_ir)
    y_pred_ir = clf.predict(X_te_ir_s)
    cm_ir = confusion_matrix(y_te_ir, y_pred_ir)
    acc_ir = accuracy_score(y_te_ir, y_pred_ir)
    results_ir.append((name, cm_ir, acc_ir))

# ── Plot ─────────────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

fig.canvas.header_visible = False
fig.canvas.toolbar_visible = True
fig.canvas.toolbar_position = 'right'

for ax, (name, cm_ir, acc_ir) in zip(axes, results_ir):
    sns.heatmap(
        cm_ir, annot=True, fmt='d', cmap='YlOrRd',
        xticklabels=class_names_ir, yticklabels=class_names_ir,
        linewidths=0.5, linecolor='white', ax=ax, annot_kws={'size': 13}
    )
    ax.set_title(f'{name}\naccuracy = {acc_ir:.1%}')
    ax.set_xlabel('Predicted class')
    ax.set_ylabel('True class')

plt.suptitle(
    'Figure 14: multi-class confusion matrices (Iris, 3 classes)\n'
    'diagonal = correct, off-diagonal = which classes were confused',
    fontsize=11, y=0.98
)
plt.tight_layout()
plt.show()

# ── Verify accuracy manually from the confusion matrix ─────────────────────
print('Manual accuracy from confusion matrix:')
for name, cm_ir, acc_ir in results_ir:
    correct = np.trace(cm_ir)   # np.trace sums the main diagonal
    total = cm_ir.sum()
    print(f'  {name}:')
    print(f'    Diagonal sum (correct): {correct}')
    print(f'    Total entries: {total}')
    print(f'    Accuracy = {correct}/{total} = {correct / total:.1%}')
    print()

---

The printed output confirms what the figure already showed, but makes the arithmetic explicit. For the deeper tree, 44 of the 45 test samples land on the diagonal of its confusion matrix, giving an accuracy of 44/45, or 97.8%; only a single sample was misclassified. For the shallow tree, only 30 of the 45 samples land on the diagonal, giving an accuracy of 30/45, or 66.7%, meaning 15 samples, exactly a third of the test set, were misclassified.

In both cases, the accuracy figure printed here matches the accuracy shown in the title of each heatmap in Figure 14, confirming that "sum of diagonal divided by sum of all entries" is exactly what `accuracy_score` was computing all along, just made visible by working through the confusion matrix directly. The shallow tree's much larger off-diagonal total, 15 misclassified samples spread across the off-diagonal cells of its confusion matrix, is where Figure 14 becomes useful beyond this single accuracy number: the heatmap shows not just how many samples were misclassified, but which classes they were confused with, information that the accuracy figure alone cannot provide.

### 9.4 Why Accuracy Lies on Imbalanced Data

Section 7.1 established, in the abstract, that accuracy can be misleading on imbalanced data: a model that simply predicts the majority class every time can achieve an accuracy equal to that class's share of the data, without learning anything at all. The confusion matrix, introduced in this section, is what makes that abstract warning concrete: it is the tool that exposes exactly how a high accuracy figure can be hiding a model that has learned nothing useful.

The key is to look beyond the overall accuracy number and examine the minority class row of the confusion matrix specifically. If a model achieves 95% accuracy on a dataset that is 95% one class and 5% the other, that 95% figure on its own tells you nothing about whether the model can recognise the minority class at all. But if you look at the minority class's row and find that most, or all, of those examples have been predicted as the majority class, that single observation tells you the model has not learned to distinguish the minority class. Its apparent "accuracy" is really just the majority class's share of the data, dressed up as a performance metric.

The code below makes this comparison directly, training a real classifier on a 95/5 imbalanced dataset and comparing it against a dummy model that always predicts the majority class and has learned nothing. Figure 15 shows that the two models' overall accuracy scores are close to each other, but places their confusion matrices side by side, so you can see that, despite the similar accuracy, the two models behave completely differently when it comes to the minority class: one of them catches some of its cases, while the other catches none at all.

In [ ]:
# Listing 18.
# ── Why accuracy misleads on imbalanced data, the confusion matrix reveals it
#
# We train a classifier on a severely imbalanced dataset (95% class 0,
# 5% class 1) and compare it against a dummy that always predicts the
# majority class.
#
# Their overall accuracy scores look similar.
# Their confusion matrices tell completely different stories.

rng_ci = np.random.default_rng(21)
X_ci_0 = rng_ci.normal([0, 0], 1.0, (950, 2))   # majority class, centred at the origin
X_ci_1 = rng_ci.normal([2, 2], 1.0, (50, 2))    # minority class, offset by (2, 2)
X_ci = np.vstack([X_ci_0, X_ci_1])
y_ci = np.array([0] * 950 + [1] * 50)

X_tr_ci, X_te_ci, y_tr_ci, y_te_ci = train_test_split(
    X_ci, y_ci, test_size=0.25, random_state=0, stratify=y_ci
)

sc_ci = StandardScaler()
X_tr_ci_s = sc_ci.fit_transform(X_tr_ci)   # fit on training data only
X_te_ci_s = sc_ci.transform(X_te_ci)       # apply same fitted scaler to test data

# Real classifier: a Decision Tree with limited depth, so it has learned
# something, but is not assumed to be perfect.
clf_ci = DecisionTreeClassifier(max_depth=3, random_state=0)
clf_ci.fit(X_tr_ci_s, y_tr_ci)
y_pred_ci = clf_ci.predict(X_te_ci_s)

# Dummy: always predicts class 0 (the majority class), learning nothing
# about the data at all.
y_dummy = np.zeros_like(y_te_ci)

acc_model = accuracy_score(y_te_ci, y_pred_ci)
acc_dummy = accuracy_score(y_te_ci, y_dummy)

cm_model = confusion_matrix(y_te_ci, y_pred_ci)
cm_dummy = confusion_matrix(y_te_ci, y_dummy)

print('Accuracy comparison:')
print(f'  Real model:       {acc_model:.1%}')
print(f'  Always-majority:  {acc_dummy:.1%}')
print()
print('They look similar. But the confusion matrices reveal the difference...')

# ── Plot ─────────────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

fig.canvas.header_visible = False
fig.canvas.toolbar_visible = True
fig.canvas.toolbar_position = 'right'

for ax, cm, title in [
    (axes[0], cm_model,
     f'Real model (acc = {acc_model:.1%})\ncatches some class 1 cases'),
    (axes[1], cm_dummy,
     f'Always predict class 0 (acc = {acc_dummy:.1%})\ncatches zero class 1 cases'),
]:
    sns.heatmap(
        cm, annot=True, fmt='d', cmap='Blues',
        xticklabels=['Pred. 0', 'Pred. 1'],
        yticklabels=['True 0', 'True 1'],
        linewidths=0.5, linecolor='white', ax=ax, annot_kws={'size': 14}
    )
    ax.set_title(title, fontsize=10)
    ax.set_xlabel('Predicted')
    ax.set_ylabel('True')

plt.suptitle(
    'Figure 15: accuracy misleads on imbalanced data\n'
    'the confusion matrix reveals what accuracy hides',
    fontsize=12, y=0.98
)
plt.tight_layout()
plt.show()

# ── Class 1 breakdown ───────────────────────────────────────────────────────
print()
print('The class 1 row tells the story:')
print(f'  Real model:      correctly identified {cm_model[1, 1]} / {y_te_ci.sum()} class 1 cases')
print(f'  Always-majority: correctly identified {cm_dummy[1, 1]} / {y_te_ci.sum()} class 1 cases')
print()
print('This is why accuracy alone is not enough for imbalanced datasets.')
print('Notebook 11 will introduce precision, recall, and F1 to address this properly.')


---

## 10. Summary

🔙 [Back to Table of Contents](#table-of-contents)

The table below summarises the key concepts covered in this notebook.

| Concept | Key Point |
|---|---|
| **Generalisation gap** | Training accuracy is always optimistic; only test accuracy on held-out data reflects true performance. |
| **i.i.d. assumption** | Training and test data must come from the same distribution; samples must be independent of each other. |
| **No Free Lunch theorem** | No single model wins on all datasets; always compare several classifiers under identical conditions. |
| **Experimental pipeline** | One fixed split, one fitted scaler, the same test set for every model in a comparison. |
| **Split ratio trade-off** | More training data improves the model; more test data improves the reliability of the accuracy estimate. |
| **Class imbalance** | Accuracy is misleading when classes are unequal; always check the class distribution before evaluating. |
| **Random sampling** | Equal probability of selection; unreliable on imbalanced datasets. |
| **Stratified sampling** | Preserves class proportions in every split; use `stratify=y` in scikit-learn. |
| **Weighted sampling** | Deliberately oversamples rare classes to balance the training distribution. |
| **k-fold cross-validation** | Averages $k$ evaluations; much more stable than a single train/test split. |
| **Choosing k** | $k = 5$ or $k = 10$ is the standard choice; always use stratified k-fold on imbalanced data. |
| **Confusion matrix** | Breaks accuracy into TP, TN, FP, FN (binary) or an $n \times n$ grid (multi-class). |
| **Accuracy formula** | $\dfrac{TP + TN}{TP + TN + FP + FN}$ (binary) or $\dfrac{\sum \text{diagonal}}{\sum \text{all entries}}$ (multi-class). |

Notebook 11 extends the confusion matrix into precision, recall, F1-score, and related metrics that give a richer picture of performance, especially on imbalanced datasets.


---

## 11. References

🔙 [Back to Table of Contents](#table-of-contents)

Pedregosa, F., Varoquaux, G., Gramfort, A., Michel, V., Thirion, B., Grisel, O., Blondel, M., Prettenhofer, P., Weiss, R., Dubourg, V., Vanderplas, J., Passos, A., Cournapeau, D., Brucher, M., Perrot, M. and Duchesnay, É. (2011) 'Scikit-learn: machine learning in Python', *Journal of Machine Learning Research*, 12, pp. 2825–2830.

Wolpert, D.H. and Macready, W.G. (1997) 'No free lunch theorems for optimization', *IEEE Transactions on Evolutionary Computation*, 1(1), pp. 67–82.

Kohavi, R. (1995) 'A study of cross-validation and bootstrap for accuracy estimation and model selection', in *Proceedings of the 14th International Joint Conference on Artificial Intelligence (IJCAI)*, Montreal, Canada, pp. 1137–1143.

Chawla, N.V., Bowyer, K.W., Hall, L.O. and Kegelmeyer, W.P. (2002) 'SMOTE: synthetic minority over-sampling technique', *Journal of Artificial Intelligence Research*, 16, pp. 321–357.

Hastie, T., Tibshirani, R. and Friedman, J. (2009) *The Elements of Statistical Learning: Data Mining, Inference, and Prediction*. 2nd edn. New York: Springer. Available at: https://hastie.su.domains/ElemStatLearn/ (Accessed: 25 May 2026).

James, G., Witten, D., Hastie, T. and Tibshirani, R. (2021) *An Introduction to Statistical Learning with Applications in R*. 2nd edn. New York: Springer. Available at: https://www.statlearning.com (Accessed: 25 May 2026).